In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path
import random
from tqdm import tqdm

# =============================================
# KONFIGURASI
# =============================================
DATASET_DIR   = "thread_data"           # folder utama dataset
TEXTURE_PATH  = "sample_test.jpg"       # path gambar tekstur kain
TARGET_COUNT  = 50                      # target jumlah citra per kelas
SEED          = 42
random.seed(SEED)
np.random.seed(SEED)

# =============================================
# LOAD TEKSTUR KAIN
# =============================================
def load_texture(texture_path):
    texture = cv2.imread(texture_path)
    if texture is None:
        raise FileNotFoundError(f"Tekstur tidak ditemukan: {texture_path}")
    texture = cv2.cvtColor(texture, cv2.COLOR_BGR2RGB)
    print(f"Tekstur berhasil dimuat: {texture.shape}")
    return texture

def apply_texture(image, texture, alpha=0.08):
    """
    Menerapkan pola tekstur kain ke citra dengan blending ringan.
    Alpha kecil (0.05-0.15) agar warna utama tetap terjaga.
    """
    h, w = image.shape[:2]
    # tile tekstur agar sesuai ukuran citra
    th, tw = texture.shape[:2]
    repeat_h = (h // th) + 2
    repeat_w = (w // tw) + 2
    tiled = np.tile(texture, (repeat_h, repeat_w, 1))
    tiled = tiled[:h, :w]
    # blend tekstur dengan citra asli
    blended = cv2.addWeighted(image.astype(np.float32), 1 - alpha,
                               tiled.astype(np.float32), alpha, 0)
    return np.clip(blended, 0, 255).astype(np.uint8)

# =============================================
# AUGMENTASI RGB
# =============================================
def augment_rgb(image, intensity=15):
    """
    Menerapkan variasi kecil pada nilai RGB.
    Setiap kanal mendapat pergeseran acak dalam rentang [-intensity, +intensity].
    """
    augmented = image.astype(np.int32)
    for ch in range(3):
        shift = random.randint(-intensity, intensity)
        augmented[:, :, ch] += shift
    return np.clip(augmented, 0, 255).astype(np.uint8)

def augment_brightness(image, factor_range=(0.85, 1.0)):
    """Variasi kecerahan keseluruhan."""
    factor = random.uniform(*factor_range)
    augmented = image.astype(np.float32) * factor
    return np.clip(augmented, 0, 255).astype(np.uint8)

def augment_channel_scale(image, scale_range=(0.90, 1.0)):
    """
    Skala independen tiap kanal RGB untuk mensimulasikan
    variasi white balance antar kamera.
    """
    augmented = image.astype(np.float32)
    for ch in range(3):
        scale = random.uniform(*scale_range)
        augmented[:, :, ch] *= scale
    return np.clip(augmented, 0, 255).astype(np.uint8)

def augment_gaussian_noise(image, std=5):
    """Tambahkan noise Gaussian ringan."""
    noise = np.random.normal(0, std, image.shape).astype(np.int32)
    augmented = image.astype(np.int32) + noise
    return np.clip(augmented, 0, 255).astype(np.uint8)

def apply_random_augmentation(image, texture):
    """
    Terapkan kombinasi augmentasi secara acak pada satu citra,
    lalu overlay tekstur kain.
    """
    # pilih 1-3 augmentasi secara acak
    aug_pool = [
        lambda img: augment_rgb(img, intensity=random.randint(1, 3)),
        lambda img: augment_brightness(img),
        lambda img: augment_channel_scale(img),
        lambda img: augment_gaussian_noise(img, std=random.randint(1, 3)),
    ]
    chosen = random.sample(aug_pool, k=random.randint(1, 3))
    result = image.copy()
    for aug_fn in chosen:
        result = aug_fn(result)

    # terapkan tekstur dengan alpha acak kecil
    alpha = random.uniform(0.04, 0.12)
    result = apply_texture(result, texture, alpha=alpha)
    return result

# =============================================
# PROSES AUGMENTASI PER KELAS
# =============================================
def augment_class(class_dir, texture, target=50):
    """
    Augmentasi satu folder kelas hingga mencapai target jumlah citra.
    Citra asli tidak dihapus.
    """
    # kumpulkan citra yang sudah ada
    valid_ext = {".jpg", ".jpeg", ".png", ".bmp"}
    existing = [f for f in Path(class_dir).iterdir()
                if f.suffix.lower() in valid_ext]
    current_count = len(existing)

    if current_count == 0:
        print(f"Tidak ada citra di {class_dir}, dilewati.")
        return 0

    if current_count >= target:
        print(f"{Path(class_dir).name}: sudah {current_count} citra, dilewati.")
        return 0

    needed = target - current_count
    generated = 0

    for i in range(needed):
        # pilih citra sumber secara acak dari yang sudah ada
        src_path = random.choice(existing)
        img = cv2.imread(str(src_path))
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # augmentasi
        aug_img = apply_random_augmentation(img_rgb, texture)

        # simpan
        aug_bgr  = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
        out_name = f"aug_{i+1:04d}.jpg"
        out_path = Path(class_dir) / out_name
        cv2.imwrite(str(out_path), aug_bgr,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        generated += 1

    return generated

# =============================================
# MAIN — JALANKAN AUGMENTASI SEMUA KELAS
# =============================================
def run_augmentation():
    # validasi path
    if not os.path.isdir(DATASET_DIR):
        raise NotADirectoryError(f"Folder dataset tidak ditemukan: {DATASET_DIR}")

    # load tekstur
    texture = load_texture(TEXTURE_PATH)

    # ambil semua subfolder kelas
    class_dirs = sorted([
        d for d in Path(DATASET_DIR).iterdir() if d.is_dir()
    ])

    if len(class_dirs) == 0:
        raise ValueError("Tidak ada subfolder kelas di dalam dataset_kain.")

    print(f"\nTotal kelas ditemukan : {len(class_dirs)}")
    print(f"Target per kelas      : {TARGET_COUNT} citra")
    print(f"{'─'*50}")

    total_generated = 0
    skipped         = 0
    already_full    = 0

    for class_dir in tqdm(class_dirs, desc="Memproses kelas"):
        existing = [f for f in class_dir.iterdir()
                    if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
        count = len(existing)

        if count == 0:
            skipped += 1
            continue
        if count >= TARGET_COUNT:
            already_full += 1
            continue

        gen = augment_class(str(class_dir), texture, target=TARGET_COUNT)
        total_generated += gen

    # =============================================
    # LAPORAN AKHIR
    # =============================================
    print(f"\n{'='*50}")
    print(f"Augmentasi selesai!")
    print(f"   Total citra dibuat    : {total_generated}")
    print(f"   Kelas sudah penuh     : {already_full}")
    print(f"   Kelas dilewati (kosong): {skipped}")
    print(f"{'='*50}")

    # verifikasi hasil
    print(f"\nVerifikasi jumlah citra per kelas:")
    print(f"{'Nama Kelas':<20} {'Jumlah Citra':>15}")
    print(f"{'─'*37}")
    for class_dir in sorted(Path(DATASET_DIR).iterdir()):
        if class_dir.is_dir():
            n = len([f for f in class_dir.iterdir()
                     if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}])
            status = "V" if n >= TARGET_COUNT else "X"
            print(f"{status} {class_dir.name:<18} {n:>10} citra")

# =============================================
# JALANKAN
# =============================================
run_augmentation()